# Inspeção do ToolACE

Objetivo: entender as descrições das ferramentas, os pedidos e as respostas supervisionadas antes de preparar o treinamento.

A revisão abaixo foi identificada na inspeção inicial no Colab e fica fixada para reprodução. O registro dessa inspeção está em `reports/data_inspection.json`.

Fonte: https://huggingface.co/datasets/Team-ACE/ToolACE — licença declarada Apache 2.0.

Executar com CPU no Colab/Kaggle. As chamadas presentes nos dados são texto; este notebook não executa as ferramentas descritas.


In [ ]:
import json
from urllib.request import urlopen

toolace_revision = "6bda777c88d21e5a204703c1ee45597a8fa4f734"
url = (
    "https://huggingface.co/datasets/Team-ACE/ToolACE/"
    f"resolve/{toolace_revision}/data.json"
)
with urlopen(url, timeout=60) as resposta:
    toolace_data = json.load(resposta)

assert isinstance(toolace_data, list) and toolace_data
exemplo = toolace_data[0]
print("Revisão:", toolace_revision)
print("Exemplos:", len(toolace_data))
print("Campos:", list(exemplo))
print("\nINSTRUÇÕES E FERRAMENTAS:\n")
print(exemplo["system"])
for papel in ("user", "assistant"):
    mensagem = next(m for m in exemplo["conversations"] if m["from"] == papel)
    print(f"\n{papel.upper()}:\n{mensagem['value']}")


## Estrutura das conversas

Verificar tipos de mensagens, sequências e comprimentos antes de selecionar o recorte. Esta auditoria não avalia a correção das chamadas. Os resultados conferidos no Colab estão no relatório de inspeção.


In [ ]:
from collections import Counter
from statistics import median

papeis = Counter()
sequencias = Counter()
tamanhos = []
problemas = []

for indice, registro in enumerate(toolace_data):
    if not isinstance(registro, dict):
        problemas.append(indice)
        continue

    mensagens = registro.get("conversations")
    estrutura_valida = (
        isinstance(registro.get("system"), str)
        and isinstance(mensagens, list)
        and len(mensagens) > 0
    )
    if not estrutura_valida:
        problemas.append(indice)
        continue

    mensagens_validas = all(
        isinstance(mensagem, dict)
        and isinstance(mensagem.get("from"), str)
        and isinstance(mensagem.get("value"), str)
        for mensagem in mensagens
    )
    if not mensagens_validas:
        problemas.append(indice)
        continue

    sequencia = tuple(mensagem["from"] for mensagem in mensagens)
    papeis.update(sequencia)
    sequencias[sequencia] += 1
    tamanhos.append(len(mensagens))

print("Registros examinados:", len(toolace_data))
print("Registros com estrutura válida:", len(tamanhos))
print("Registros com problemas estruturais:", len(problemas))
print("\nQuantidade de mensagens por papel:")
for papel, quantidade in papeis.most_common():
    print(f"  {papel}: {quantidade}")
print("\nOito sequências de conversa mais frequentes:")
for sequencia, quantidade in sequencias.most_common(8):
    print(f"  {' → '.join(sequencia)}: {quantidade}")
if tamanhos:
    print("\nMensagens por conversa:")
    print("  Mínimo:", min(tamanhos))
    print("  Mediana:", median(tamanhos))
    print("  Máximo:", max(tamanhos))
if problemas:
    print("\nPrimeiros índices com problemas:", problemas[:10])


## Leitura das descrições de ferramentas

O primeiro colchete das instruções nem sempre inicia a lista de ferramentas: pode pertencer a um parâmetro interno. Esta etapa usa o marcador explícito da apresentação JSON. Registros com outras representações ficam fora dos candidatos atuais, sem serem classificados como inválidos. A separação das respostas por colchetes é apenas uma observação de formato.


In [ ]:
import json
from collections import Counter

marcador = "Here is a list of functions in JSON format that you can invoke:"
candidatos_json = []
diagnostico = Counter()
formatos_resposta = Counter()
erros_json = []

for indice, registro in enumerate(toolace_data):
    mensagens = registro["conversations"]
    if [m["from"] for m in mensagens] != ["user", "assistant"]:
        continue
    instrucoes = registro["system"]
    if marcador not in instrucoes:
        diagnostico["Sem o marcador esperado"] += 1
        continue
    trecho = instrucoes.split(marcador, 1)[1].lstrip()
    try:
        ferramentas, _ = json.JSONDecoder().raw_decode(trecho)
        if not isinstance(ferramentas, list) or not ferramentas:
            raise ValueError("Lista vazia ou formato diferente de lista")
        for ferramenta in ferramentas:
            if not isinstance(ferramenta, dict):
                raise ValueError("Ferramenta não é um objeto")
            if not isinstance(ferramenta.get("name"), str):
                raise ValueError("Nome ausente ou inválido")
            if not isinstance(ferramenta.get("parameters"), dict):
                raise ValueError("Parâmetros ausentes ou inválidos")
    except (ValueError, TypeError) as erro:
        diagnostico["Marcador presente, mas leitura falhou"] += 1
        erros_json.append((indice, str(erro)))
        continue
    resposta = mensagens[1]["value"].strip()
    candidatos_json.append({
        "indice_original": indice,
        "ferramentas": ferramentas,
        "pedido": mensagens[0]["value"],
        "resposta": resposta,
    })
    diagnostico["Ferramentas lidas com sucesso"] += 1
    formato = (
        "Resposta entre colchetes"
        if resposta.startswith("[") and resposta.endswith("]")
        else "Outro formato de resposta"
    )
    formatos_resposta[formato] += 1

print("Diagnóstico das 10.500 conversas curtas:")
for motivo, quantidade in diagnostico.items():
    print(f"  {motivo}: {quantidade}")
print("\nFormato das respostas nos candidatos:")
for formato, quantidade in formatos_resposta.items():
    print(f"  {formato}: {quantidade}")
if erros_json:
    print("\nPrimeiras falhas:", erros_json[:5])


## Amostra de respostas fora dos colchetes

Amostragem com semente fixa para investigar recusas por falta de ferramenta, pedidos de dados faltantes e outras possíveis representações. Os quatro casos inspecionados não definem rótulos para toda a população.


In [ ]:
import random

respostas_outros_formatos = [
    exemplo
    for exemplo in candidatos_json
    if not (
        exemplo["resposta"].startswith("[")
        and exemplo["resposta"].endswith("]")
    )
]
amostra = random.Random(42).sample(
    respostas_outros_formatos,
    min(4, len(respostas_outros_formatos)),
)
print("Total de respostas fora dos colchetes:", len(respostas_outros_formatos))
for exemplo in amostra:
    print("\n" + "=" * 60)
    print("ÍNDICE ORIGINAL:", exemplo["indice_original"])
    print("\nFERRAMENTAS DISPONÍVEIS:")
    for ferramenta in exemplo["ferramentas"]:
        print("-", ferramenta["name"])
    print("\nPEDIDO:")
    print(exemplo["pedido"][:600])
    print("\nRESPOSTA ANOTADA:")
    print(exemplo["resposta"][:1400])
    if len(exemplo["pedido"]) > 600 or len(exemplo["resposta"]) > 1400:
        print("\n[Exibição limitada; o conteúdo original foi preservado.]")


## Leitura de chamadas únicas


In [ ]:
import ast

def ler_chamada_unica(resposta, ferramentas):
    interior = resposta.strip()[1:-1].strip()
    for ferramenta in ferramentas:
        nome = ferramenta["name"]
        if not interior.startswith(nome + "("):
            continue
        try:
            texto = "funcao" + interior[len(nome):]
            chamada = ast.parse(texto, mode="eval").body
            if not (
                isinstance(chamada, ast.Call)
                and isinstance(chamada.func, ast.Name)
                and chamada.func.id == "funcao"
            ):
                return None
            if chamada.args:
                return None
            if any(argumento.arg is None for argumento in chamada.keywords):
                return None
            argumentos = {
                argumento.arg: ast.literal_eval(argumento.value)
                for argumento in chamada.keywords
            }
            if len(argumentos) != len(chamada.keywords):
                return None
            return {"ferramenta": nome, "argumentos": argumentos}
        except (SyntaxError, ValueError, TypeError, RecursionError):
            return None
    return None

chamadas_unicas = []
nao_interpretadas = []
for exemplo in candidatos_json:
    resposta = exemplo["resposta"]
    if not (resposta.startswith("[") and resposta.endswith("]")):
        continue
    chamada = ler_chamada_unica(resposta, exemplo["ferramentas"])
    if chamada is None:
        nao_interpretadas.append(exemplo)
    else:
        chamadas_unicas.append({**exemplo, "chamada": chamada})
print("Respostas entre colchetes examinadas:", len(chamadas_unicas) + len(nao_interpretadas))
print("Chamadas únicas legíveis:", len(chamadas_unicas))
print("Não interpretadas neste recorte:", len(nao_interpretadas))
if chamadas_unicas:
    exemplo = chamadas_unicas[0]
    print("Índice:", exemplo["indice_original"])
    print(exemplo["chamada"])
for exemplo in nao_interpretadas[:3]:
    print("Índice:", exemplo["indice_original"])
    print(exemplo["resposta"][:500])


## Nomes dos argumentos e campos obrigatórios


In [ ]:
from collections import Counter

verificacao_argumentos = Counter()
chamadas_compativeis = []
casos_para_revisao = []
for exemplo in chamadas_unicas:
    chamada = exemplo["chamada"]
    correspondencias = [f for f in exemplo["ferramentas"] if f["name"] == chamada["ferramenta"]]
    if len(correspondencias) != 1:
        verificacao_argumentos["Nome sem correspondência única"] += 1
        continue
    parametros = correspondencias[0]["parameters"]
    propriedades = parametros.get("properties")
    obrigatorios = parametros.get("required", [])
    esquema_legivel = (
        isinstance(propriedades, dict)
        and isinstance(obrigatorios, list)
        and all(isinstance(nome, str) for nome in obrigatorios)
    )
    if not esquema_legivel:
        verificacao_argumentos["Esquema fora do formato esperado"] += 1
        continue
    fornecidos = set(chamada["argumentos"])
    faltantes = set(obrigatorios) - fornecidos
    fora_dos_campos = fornecidos - set(propriedades)
    if faltantes or fora_dos_campos:
        verificacao_argumentos["Divergência nos argumentos"] += 1
        casos_para_revisao.append({
            "indice": exemplo["indice_original"], "ferramenta": chamada["ferramenta"],
            "obrigatorios_ausentes": sorted(faltantes), "argumentos_fora_dos_campos": sorted(fora_dos_campos),
        })
    else:
        verificacao_argumentos["Compatíveis nos campos verificados"] += 1
        chamadas_compativeis.append(exemplo)
print("Chamadas examinadas:", len(chamadas_unicas))
for resultado, quantidade in verificacao_argumentos.items():
    print(f"  {resultado}: {quantidade}")
for caso in casos_para_revisao[:5]:
    print(caso)


## Duplicação exata e tamanho em caracteres


In [ ]:
import json
import hashlib
from collections import defaultdict, Counter
from statistics import median

def serializar_entrada(exemplo):
    ferramentas_ordenadas = sorted(
        exemplo["ferramentas"],
        key=lambda ferramenta: json.dumps(ferramenta, sort_keys=True, ensure_ascii=False),
    )
    return json.dumps(
        {"ferramentas": ferramentas_ordenadas, "pedido": exemplo["pedido"].strip()},
        sort_keys=True, ensure_ascii=False, separators=(",", ":"),
    )

textos_candidatos = [
    e for e in candidatos_json
    if not (e["resposta"].startswith("[") and e["resposta"].endswith("]"))
]
base_candidata = [
    {**e, "formato": "chamada_unica"} for e in chamadas_compativeis
] + [
    {**e, "formato": "fora_dos_colchetes"} for e in textos_candidatos
]
por_entrada = defaultdict(list)
for exemplo in base_candidata:
    entrada = serializar_entrada(exemplo)
    chave = hashlib.sha256(entrada.encode("utf-8")).hexdigest()
    por_entrada[chave].append(exemplo)
exemplos_preparados = []
entradas_para_revisao = []
duplicatas_unificadas = 0
for chave, grupo in por_entrada.items():
    respostas = {e["resposta"].strip() for e in grupo}
    if len(respostas) > 1:
        entradas_para_revisao.append({
            "id_entrada": chave,
            "indices_originais": [e["indice_original"] for e in grupo],
            "quantidade_respostas_distintas": len(respostas),
        })
        continue
    exemplos_preparados.append({
        **grupo[0], "id_entrada": chave,
        "indices_originais": [e["indice_original"] for e in grupo],
    })
    duplicatas_unificadas += len(grupo) - 1
tamanhos_caracteres = sorted(
    len(serializar_entrada(e)) + len(e["resposta"]) for e in exemplos_preparados
)
print("Candidatos antes da conferência:", len(base_candidata))
print("Duplicatas com mesma resposta unificadas:", duplicatas_unificadas)
print("Entradas com respostas diferentes:", len(entradas_para_revisao))
print("Exemplos preparados:", len(exemplos_preparados))
print(Counter(e["formato"] for e in exemplos_preparados))
if tamanhos_caracteres:
    print("Mínimo:", tamanhos_caracteres[0])
    print("Mediana:", median(tamanhos_caracteres))
    print("Máximo:", tamanhos_caracteres[-1])
for entrada in entradas_para_revisao[:3]:
    print(entrada)


## Separação por grupos e exportação


In [ ]:
import json
import random
import hashlib
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter

SEMENTE = 42
PASTA = Path("/content/triunfo_data")
assert toolace_revision == "6bda777c88d21e5a204703c1ee45597a8fa4f734"
assert len(exemplos_preparados) == 5586

def hash_texto(texto):
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

def chaves_agrupamento(exemplo):
    pedido = unicodedata.normalize("NFKC", exemplo["pedido"])
    pedido = " ".join(pedido.casefold().split())
    ferramentas = sorted(
        json.dumps(f, sort_keys=True, ensure_ascii=False)
        for f in exemplo["ferramentas"]
    )
    return (
        ("pedido", hash_texto(pedido)),
        ("ferramentas", hash_texto(json.dumps(ferramentas))),
    )

pais = list(range(len(exemplos_preparados)))
def encontrar(indice):
    while pais[indice] != indice:
        pais[indice] = pais[pais[indice]]
        indice = pais[indice]
    return indice

primeiro_por_chave = {}
for indice, exemplo in enumerate(exemplos_preparados):
    for chave in chaves_agrupamento(exemplo):
        if chave in primeiro_por_chave:
            raiz_atual = encontrar(indice)
            raiz_anterior = encontrar(primeiro_por_chave[chave])
            pais[raiz_atual] = raiz_anterior
        else:
            primeiro_por_chave[chave] = indice
por_grupo = defaultdict(list)
for indice, exemplo in enumerate(exemplos_preparados):
    por_grupo[encontrar(indice)].append(exemplo)
grupos = list(por_grupo.values())
print("Grupos encontrados:", len(grupos))
print("Maiores grupos:", sorted((len(g) for g in grupos), reverse=True)[:5])
assert len(grupos) >= 3, "Grupos insuficientes. Compartilhe a saída."
random.Random(SEMENTE).shuffle(grupos)
grupos.sort(key=len, reverse=True)
partes = {"train": [], "validation": [], "test": []}
proporcoes = {"train": 0.8, "validation": 0.1, "test": 0.1}
total = len(exemplos_preparados)
for posicao, grupo in enumerate(grupos):
    vazios = [nome for nome in partes if not partes[nome]]
    grupos_restantes = len(grupos) - posicao
    if vazios and grupos_restantes == len(vazios):
        destino = vazios[0]
    else:
        destino = max(partes, key=lambda nome: proporcoes[nome] * total - len(partes[nome]))
    partes[destino].extend(grupo)
ids = {nome: {e["id_entrada"] for e in registros} for nome, registros in partes.items()}
chaves = {
    nome: {chave for e in registros for chave in chaves_agrupamento(e)}
    for nome, registros in partes.items()
}
for esquerda, direita in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert ids[esquerda].isdisjoint(ids[direita])
    assert chaves[esquerda].isdisjoint(chaves[direita])
assert sum(map(len, partes.values())) == total
assert all(partes.values())
PASTA.mkdir(parents=True, exist_ok=True)
arquivos = {}
for nome, registros in partes.items():
    registros.sort(key=lambda e: e["id_entrada"])
    caminho = PASTA / f"{nome}.jsonl"
    with caminho.open("w", encoding="utf-8") as arquivo:
        for exemplo in registros:
            arquivo.write(json.dumps(exemplo, ensure_ascii=False) + "\n")
    arquivos[nome] = {
        "exemplos": len(registros),
        "formatos": dict(Counter(e["formato"] for e in registros)),
        "sha256": hashlib.sha256(caminho.read_bytes()).hexdigest(),
    }
manifesto = {
    "dataset": "Team-ACE/ToolACE", "dataset_revision": toolace_revision,
    "seed": SEMENTE, "total_examples": total, "group_count": len(grupos),
    "split_policy": "Grupos conectados por pedido normalizado igual ou conjunto idêntico de descrições de ferramentas; proporção aproximada 80/10/10.",
    "limitations": [
        "Ferramentas individuais podem aparecer em divisões diferentes.",
        "A checagem não identifica todas as paráfrases ou semelhanças semânticas.",
        "Tamanho em tokens e compatibilidade com o treinamento ainda serão medidos.",
        "O formato fora_dos_colchetes não é um rótulo semântico de recusa.",
    ],
    "source": "https://huggingface.co/datasets/Team-ACE/ToolACE",
    "declared_data_license": "Apache-2.0", "files": arquivos,
}
(PASTA / "manifest.json").write_text(
    json.dumps(manifesto, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
for nome, info in arquivos.items():
    print(nome, info["exemplos"], info["formatos"])
print("Sobreposições verificadas: nenhuma.")
print("Pasta:", PASTA)
print(json.dumps(manifesto, ensure_ascii=False, indent=2))


## Checkpoint

As contagens e hashes observados no Colab estão em `reports/dataset_manifest.json`. As oito células da versão consolidada foram reexecutadas; os hashes de treino, validação e teste coincidiram com os arquivos do Colab. A conferência está em `reports/reproduction_check.json`. Não há treino ou resultados de modelo neste checkpoint.
